<a href="https://colab.research.google.com/github/jach36/EEG-mindstorm-bci/blob/main/Brain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

In [ ]:
# install dependencies
!pip install mne scipy numpy matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 60.7 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.


In [11]:
# All import statements needed
# Run this!!
import mne
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import os

# Preprocessing EEG data

In [ ]:

# load eeg dataset
data = sio.loadmat('/content/drive/MyDrive/BCI/s01.mat')
eeg = data["eeg"]

# check sampling rate
srate = eeg['srate'][0,0].flatten()[0]
print("Sampling rate: ", srate, "Hz")

# Unwrap MATLAB format ([0,0])
left = eeg['movement_left'][0,0]
right = eeg['movement_right'][0,0]

n_left_trials = eeg['n_imagery_trials'][0,0].flatten()[0]

timepoints_per_trial = 71680 / n_left_trials

# Sliced 1 trial [channels, timepoints] ([:,0:716])
left_trial_one = left[:,0:716]
print("Trial one: ",left_trial_one.shape)

# Get all trials
trials_left = []
trials_right = []
start = 0
end = 716

for i in range(100):
  one_trial = left[:, start:end]
  trials_left.append(one_trial)
  one_trial = right[:, start:end]
  trials_right.append(one_trial)
  start += 716
  end += 716

trials_left = np.array(trials_left)
trials_right = np.array(trials_left)
print("shape left: ", trials_left.shape)
print("shape right: ", trials_right.shape)

# Stacking and labeling
X = np.vstack([trials_left, trials_right])
# Corresponding labels
Y = np.array([0] * 100 + [1] * 100)

print("y shape : ", X.shape)




# Plotting Brain Data

In [ ]:

from scipy.signal import butter, filtfilt

c3_index = 7

# [trial 0, channel index 7, all timepoints]
trial = X[0, c3_index, :]

# Time axis in seconds
time = np.linspace(0, 716/512, 716)


# Bandpass filter
def bandpass_filter(data, lowcut=8, highcut=30, fs=512, order=5):
    nyq = fs / 2                        # Nyquist frequency
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

trial_filtered = bandpass_filter(trial)

# Plot both raw and filtered side by side
fig, axes = plt.subplots(2, 1, figsize=(10, 6))

axes[0].plot(time, trial, color='gray')
axes[0].set_title("Raw EEG — C3, left hand trial 1")
axes[0].set_ylabel("Amplitude")

axes[1].plot(time, trial_filtered, color='steelblue')
axes[1].set_title("Filtered EEG (8–30 Hz) — C3, left hand trial 1")
axes[1].set_ylabel("Amplitude")
axes[1].set_xlabel("Time (seconds)")
plt.tight_layout()
plt.show()



In [ ]:
# Plot Power Spectrum
from scipy.signal import welch

# Compute power spectrum for raw and filtered
freqs_raw, power_raw = welch(trial, fs=512, nperseg=256)
freqs_filt, power_filt = welch(trial_filtered, fs=512, nperseg=256)

plt.figure(figsize=(10, 4))
plt.semilogy(freqs_raw, power_raw, color='gray', label='Raw')
plt.semilogy(freqs_filt, power_filt, color='steelblue', label='Filtered')
plt.axvspan(8, 30, alpha=0.2, color='green', label='Our band (8–30 Hz)')
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power")
plt.title("Power spectrum — raw vs filtered")
plt.legend()
plt.xlim(0, 60)
plt.show()